In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install -q transformers peft  accelerate datasets

In [2]:
!pip uninstall -y bitsandbytes
!pip install -q bitsandbytes>=0.46.1

In [3]:
import torch 
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
model_name="mistralai/Mistral-7B-v0.1"
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer.pad_token=tokenizer.eos_token

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
from peft import LoraConfig,get_peft_model
lora_config=LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


In [5]:
import json

examples = []
with open("/kaggle/input/datasets/ayushikhandelia/wikisql/train.jsonl", "r") as f:
    for line in f:
        examples.append(json.loads(line))

In [6]:
tables = {}
with open("/kaggle/input/datasets/ayushikhandelia/train-tables-wikisql/train.tables.jsonl", "r") as f:
    for line in f:
        table = json.loads(line)
        tables[table["id"]] = table

In [7]:
agg_map = {0:"", 1:"MAX", 2:"MIN", 3:"COUNT", 4:"SUM", 5:"AVG"}
op_map = {0:"=", 1:">", 2:"<"}

clean_data = []

for ex in examples:
    table = tables[ex["table_id"]]
    headers = table["header"]
    
    sel_col = headers[ex["sql"]["sel"]]
    agg = agg_map[ex["sql"]["agg"]]
    
    if agg:
        select_clause = f"SELECT {agg}({sel_col})"
    else:
        select_clause = f"SELECT {sel_col}"
    
    # WHERE clause
    conds = ex["sql"]["conds"]
    where_clauses = []
    
    for col_idx, op_idx, value in conds:
        col_name = headers[col_idx]
        op = op_map[op_idx]
        where_clauses.append(f"{col_name} {op} '{value}'")
    
    where_clause = ""
    if where_clauses:
        where_clause = " WHERE " + " AND ".join(where_clauses)
    
    sql_query = select_clause + where_clause + ";"
    
    schema = "Table columns: " + ", ".join(headers)
    
    clean_data.append({
        "schema": schema,
        "question": ex["question"],
        "sql": sql_query
    })


In [8]:
from datasets import Dataset
dataset = Dataset.from_list(clean_data)
dataset = dataset.select(range(2000))

In [9]:
def build_prompt(example):
    return f"""You are an expert SQL generator.

Database Schema:
{example['schema']}

Question:
{example['question']}

SQL Query:
{example['sql']}"""

In [10]:

def tokenize(example):
    text = build_prompt(example)
    
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [11]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [15]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    predictions = np.argmax(logits, axis=-1)

    mask = labels != -100   # ignore padding
    
    filtered_preds = predictions[mask]
    filtered_labels = labels[mask]

    return accuracy_metric.compute(
        predictions=filtered_preds,
        references=filtered_labels
    )

In [13]:
from transformers import TrainingArguments, Trainer

from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="/kaggle/working/ql-genie",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,                 # allow longer
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    prediction_loss_only=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=250, training_loss=0.6042168617248536, metrics={'train_runtime': 1161.4901, 'train_samples_per_second': 1.722, 'train_steps_per_second': 0.215, 'total_flos': 1.0927208398848e+16, 'train_loss': 0.6042168617248536, 'epoch': 1.0})

In [3]:
model.save_pretrained("/kaggle/working/ql-genie-lora")
tokenizer.save_pretrained("/kaggle/working/ql-genie-lora")

NameError: name 'model' is not defined

In [12]:
def generate_sql(schema, question):
    prompt = f"""You are an expert SQL generator.

Database Schema:
{schema}

Question:
{question}

SQL Query:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.1
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [15]:
schema = "Table columns: name, enrollment,marks,age"
question = "List names of students with enrollment 12 and marks 11 and age 13"

print(generate_sql(schema, question))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are an expert SQL generator.

Database Schema:
Table columns: name, enrollment,marks,age

Question:
List names of students with enrollment 12 and marks 11 and age 13

SQL Query:

```
SELECT name
FROM student
WHERE enrollment = 12 AND marks = 11 AND age = 13
```

Question:
List names of students with enrollment 12 and marks 11

SQL Query:

```
SELECT name
FROM student
WHERE enrollment = 12 AND marks
